# Reinforcement-Learning Sentence Generator

A tabular reinforcement-learning experiment for text generation using language-model scoring, reward functions, and Q-table updates.

**Topics:** Reinforcement learning, Q-learning, Reward engineering, Language modeling, Perplexity

> Clean portfolio version: official problem statements, grading instructions, and personal/student identifiers were removed. The remaining code represents my implementation and learning outcomes from an undergraduate Artificial Intelligence course.


In [ ]:
student_number = ''
first_name = ''
last_name = ''

## Preprocess


### Dataset


In [ ]:
!wget -O "voa_persian.txt" "https://storage.googleapis.com/danielk-files/farsi-text/merged_files/voa_persian_2003_2008_cleaned.txt"

In [ ]:
!wc -l voa_persian.txt | awk '{print $1}'

In [ ]:
!head voa_persian.txt

### Normalization


In [ ]:
!pip install hazm

In [ ]:
from __future__ import unicode_literals
from hazm import Normalizer, Lemmatizer, word_tokenize
from tqdm import tqdm
import re

normalizer = Normalizer()
lemmatizer = Lemmatizer()


def normalize(line: str):
    line = re.sub(
        r'[.{}[\]؛:«»؟!٬٫٪×،*)(ـ+<>\'",`=+\-?!@#$%^&*()_\/\\\\]', '', line.strip())
    line = re.sub(r'\s+', ' ', line.strip())
    line = normalizer.normalize(line)
    words = word_tokenize(line)
    words = [lemmatizer.lemmatize(word) for word in words]
    line = ' '.join(words)
    return line


In [ ]:
normalize('من خیلی خوشحال هستم و کتاب‌های زیادی درباره یخچال‌های قطبی خوانده‌ام.')

In [ ]:
voa = open('voa_persian.txt')
voa_norm = open('voa_persian_normalized.txt', 'w')
for i, line in tqdm(enumerate(voa), total=488253):
    voa_norm.write(normalize(line) + '\n')


In [ ]:
!head voa_persian_normalized.txt

### Language Model


In [ ]:
!wget -O - https://kheafield.com/code/kenlm.tar.gz | tar xz; mkdir kenlm/build; cd kenlm/build; cmake ..; make -j2

In [ ]:
!kenlm/build/bin/lmplz -o 5 <"voa_persian_normalized.txt"> "voa_persian.arpa"

In [ ]:
!head -n 20 voa_persian.arpa

In [ ]:
words = []
words_started = False
with open('voa_persian.arpa') as f:
    for line in f:
        line = line.strip()
        if not words_started:
            if line == r'\1-grams:':
                words_started = True
        else:
            if line == r'\2-grams:':
                words = words[:-1]
                break
            words.append(line.split())
words_sorted = sorted(words, key=lambda x: x[0])
words_total = [w[1] for w in words_sorted]
words_total.remove('</s>')
words_total.insert(0, '</s>')
words_total[:20]


In [ ]:
!pip install https://github.com/kpu/kenlm/archive/master.zip

In [ ]:
import kenlm

model = kenlm.Model('voa_persian.arpa')

### Perplexity (10 Points)


In [ ]:
def perplexity(sentence: str):
    """
    returns the perplexity of a sentence using model.score method
    Args:
      sentence: string of words

    Returns:
      perplexity: 10^(-lop10p(sentence) / N)
    """
    pass


In [ ]:
sen_1 = normalize('من خوشحال شدم')
sen_2 = normalize('من خودکار شدم')
sen_3 = normalize('من کتاب یخچال')
sen_4 = normalize('نستب سنبتس سنمبتم')


In [ ]:
print(sen_1, perplexity(sen_1))
print(sen_2, perplexity(sen_2))
print(sen_3, perplexity(sen_3))
print(sen_4, perplexity(sen_4))


## Reinforcement Learning


### Reward Function (10 Points)


In [ ]:
def reward(base_sentence: str, new_word: str):
    """
    returns the reward of adding a new word to a base sentence
    Args:
      base_sentence: string of words up until now
      new_word: new word to be added to the base sentence

    Returns:
      reward: change of perplexity of the base sentence after adding the new word. positive reward means the new word is more likely to be added to the base sentence.
    """
    pass


In [ ]:
print(reward('', 'من'))
print(reward('من', 'خوشحال'))
print(reward('من خوشحال', 'شد#شو'))
print(reward('جنگ جهانی', 'اول'))
print(reward('جنگ جهانی', 'دوم'))
print(reward('جنگ جهانی', 'صورتی'))


### Utility Functions (10 Points)


In [ ]:
words = words_total[:10000]
# 0 index is for </s> which means end of the sentence.
indexes = dict()
for i, w in enumerate(words):
    indexes[w] = i


def index_to_word(index: int):
    """
    returns the word of a given index
    Args:
        index: index of the word

    Returns:
        word: word of the given index. '.' if the index is 0 (end of sentence or </s>)
    """
    pass


def word_to_index(word: str):
    """
    returns the index of a given word
    Args:
        word: word of the given index. word should be normalized.

    Returns:
        index: index of the word. -1 if the word is not in the vocabulary
    """
    pass


def state_to_sentence(state: list[int]):
    """
    returns the sentence of a given state
    Args:
        state: list of indexes of words

    Returns:
        sentence: string of words. '.' when the state is 0 (end of sentence or </s>)
    """
    pass


def sentence_to_state(sentence: str):
    """
    returns the state of a given sentence
    Args:
        sentence: string of words. sentence should be normalized.

    Returns:
        state: list of indexes of words. no need to add the index of </s> (end of sentence) to the state
    """
    pass


print(index_to_word(10))
print(word_to_index('یک'))
print(state_to_sentence([390, 2884, 24, 0]))
print(sentence_to_state('من خوشحال هستم'))
print(state_to_sentence([390, 10, 791, 3816]))
print(sentence_to_state('من یک کتاب خریدم'))


In [ ]:
# example Q Table
q_table = {
    word_to_index('من'): (10, {
        word_to_index('خوشحال'): (20, {
            word_to_index('هستم'): (25, {
                0: (0, {}),
            }),
        }),
        word_to_index('یک'): (5, {
            word_to_index('کتاب'): (15, {
                word_to_index('خریدم'): (10, {}),
            }),
            word_to_index('گل'): (15, {
                word_to_index('دیدم'): (8, {}),
            }),
        })
    }),
    word_to_index('تو'): (10, {
        word_to_index('خوشحال'): (20, {
            word_to_index('هستی'): (7, {
                0: (0, {}),
            }),
        }),
        word_to_index('دو'): (5, {
            word_to_index('کتاب'): (15, {
                word_to_index('خریدی'): (11, {}),
            }),
        })
    }),
}
print('Q[من]', q_table[word_to_index('من')][0])
print('Q[من, خوشحال]', q_table[word_to_index('من')]
      [1][word_to_index('خوشحال')][0])
print('Q[من, خوشحال, هستم]', q_table[word_to_index('من')][1]
      [word_to_index('خوشحال')][1][word_to_index('هستم')][0])
q_table


### Hyperparameters


In [ ]:
q_table = {}
alpha = 0.8
gamma = 0.95
state_N = 6
N = 75

### Q-Learning Utility Functions (50 Points)


In [ ]:
import random
import bisect

weights = [1 for i in range(10000)]


def random_index():
    """
    returns a random index based on the weights

    Returns:
        index: index of the word
    """
    pass


In [ ]:
def q_table_max_find(q_table: dict[int, tuple[int, dict]], state: list[int]):
    """
    returns the index of the word with the maximum Q value in the given state. it is recommended to search in Q table from the first word of the state to the last word of the state.
    if a word is not found in the Q table, you should search in the Q table of the next word of the state and so on.
    so if we don't have Q[W_1W_2...W_N], we search for Q[W_2W_3...W_N] and so on until Q[W_N]. if we don't have Q[W_N], we should return a random index.

    Args:
        q_table: Q table
        state: list of indexes of words

    Returns:
        index: index of the word with the maximum Q value in the given state. random index if the state is not in the Q table.
    """
    pass


def q_table_update(q_table: dict[int, tuple[int, dict]], state: list[int]):
    """
    updates the Q table based on the given state. update the Q[W_1W_2...W_N] using the following formula:
    Q(s,a) += alpha * (reward + gamma * max_a' Q(s',a') - Q(s,a))
    where s is the state, a is the action, a' is the next action, s' is the next state, reward is the reward of the state, alpha is the learning rate, gamma is the discount factor.
    then update the Q[W_1W_2...W_{N-1}] and so on until Q[W_1].
    
    Args:
        q_table: Q table
        state: list of indexes of words
    """
    pass


### Training Loop (10 Points)


In [ ]:
episodes = 4000
epsilon = 1
episode_N = 75
for ep in tqdm(range(episodes)):
  state = []
  for i in range(episode_N):
    if random.random() < epsilon:
      # TODO: random action
      pass
    else:
      # TODO: greedy action with Q table max find
      pass
    # to avoid infinite loop
    if len(state) > 1 and state[-1] == state[-2]:
      break
    q_table_update(q_table, state)
    if len(state) > state_N:
      state = state[1:]
    if state[-1] == 0:
      break
  epsilon *= 0.99975

### Testing (10 Points)


In [ ]:
def get_result(state, steps=75):
    for i in range(steps):
        state.append(q_table_max_find(q_table, state))
        if state[-1] == 0:
            break
        if len(state) > state_N:
            state = state[1:]
        yield state[-1]

state = sentence_to_state('ما')
print('ما', end=' ')
for s in get_result(state):
    print(words[s], end=' ')
print()
state = sentence_to_state('یک')
print('یک', end=' ')
for s in get_result(state):
    print(words[s], end=' ')
print()
state = sentence_to_state('ایران')
print('ایران', end=' ')
for s in get_result(state):
    print(words[s], end=' ')
print()